In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

dbutils.widgets.text("extractor_notebook_path", "")
dbutils.widgets.text("config_path", "")
dbutils.widgets.text("timeout", "")

extractor_notebook_path = dbutils.widgets.get("extractor_notebook_path")
config_path = dbutils.widgets.get("config_path")
timeout = dbutils.widgets.get("timeout")

with open(config_path) as f:
    triggers = json.load(f)

In [ ]:
def run_worker(trigger):
    return dbutils.notebook.run(
        "./worker_nrt",
        int(timeout) + 600,
        {
            "extractor_notebook_path": extractor_notebook_path,
            "trigger": json.dumps(trigger),
            "timeout": timeout,
        },
    )

results = {}
with ThreadPoolExecutor(max_workers=min(len(triggers), 8)) as executor:
    futures = {executor.submit(run_worker, trigger): trigger for trigger in triggers}
    for future in as_completed(futures):
        trigger = futures[future]
        results[json.dumps(trigger["func_params"])] = json.loads(future.result())

In [ ]:
updated_triggers = [result["trigger"] for result in results.values()]

with open(config_path, "w") as f:
    json.dump(updated_triggers, f, indent=2)

In [ ]:
failed = [key for key, result in results.items() if result["status"] != 1]
if failed:
    raise Exception(f"nrt ingestion failed for: {failed}")